# G1M1: Baseline (I²) vs Cube-Root vs Sigmoid — Unified Comparator

Compare three voltage models that differ only in the **4th (nonlinear) term**:

| Model | Nonlinear term | #Params | Notes |
|-------|---------------|---------|-------|
| **Baseline (I²)** | $c_4 \cdot I^2_s$ | 5 | Linear in parameters |
| **Cube-root** | $c_4 \cdot (I - c_6)^{1/3}$ | 6 | Nonlinear, `curve_fit` |
| **Sigmoid** | $c_4 \cdot \sigma(I - c_6)$ | 6 | Nonlinear, `curve_fit` |

Goals:
- Use `UnifiedModelComparator` for systematic comparison across Low/Medium/High reference conditions.
- Include GT metrics where available.
- Provide cross-model diagnostics (fit quality, coefficient evolution, coverage).

In [4]:
import pandas as pd
from pathlib import Path
from IPython.display import display, Markdown

from degradation_toolbox.Urc.Urc1 import Urc1
from degradation_toolbox.Urc.Urc1_c6_sigmoid import Urc1_c6_sigmoid
from master_arbeit_Di.explore.UnifiedModelComparator import UnifiedModelComparator
from master_arbeit_Di.explore.GMpreprocess import GMpreprocess

In [7]:
# =============================================================================
# CONFIGURATION SECTION - EDIT THESE TO CUSTOMIZE YOUR ANALYSIS
# =============================================================================
# Dataset and directories
DATASET_PATH = r"..\\..\\explore_data\\G1M1_new.parquet"
PREPROCESS_OUTPUT_DIR = r"..\\..\\explore_data\\output"
PLOTS_OUTPUT_DIR = r"..\\plots\\c6\\G1M1_comparison"
SAVE_PLOTS = True

# Reference condition configurations: Low, Medium, High
REF_CONFIGS = {
    "Low": {
        "Iref": 0.3,
        "Tref": 58,
        "OHref": 11,
        "gt_file": r"..\\ground_truth\\output_backup\\gt_raw_processed\\G1M1_new__gt_diagram__g1m1_low_load__iref_0p3__tref_58__ohref_11__daily_regression_full_coverage.csv",
    },
    "Medium": {
        "Iref": 1.0,
        "Tref": 58,
        "OHref": 100,
        "gt_file": r"..\\ground_truth\\output_backup\\gt_raw_processed\\G1M1_new__gt_diagram__g1m1_mid_load__iref_1__tref_58__ohref_100__daily_regression_full_coverage.csv",
    },
    "High": {
        "Iref": 1.48,
        "Tref": 57,
        "OHref": 100,
        "gt_file": r"..\\ground_truth\\output_backup\\gt_raw_processed\\G1M1_new__gt_diagram__g1m1_high_load__iref_1p48__tref_57__ohref_100__daily_regression_full_coverage.csv",
    },
}

# Shared model config
COMMON_CONFIG = {
    "Iref": [cfg["Iref"] for cfg in REF_CONFIGS.values()],
    "Tref": 60,
    "OHref": 72,
    "ref_config": REF_CONFIGS,
    "len_interval": 2,
    "slide": 1,
    "min_num_data_required_for_fit": 300,
    "threshold": 1e6,
    "i_off": 0.1,
    "u_off": 1.3,
    "plot_fit": 0,
    "data_filter_i_min": 0.1,
    "data_filter_U_min": 1.4,
    "data_filter_U_max": 2.3,
    "data_filter_T_min": 50,
    "data_filter_T_max": 65,
}

# Comparison settings
SHOW_GT_METRICS = True
SHOW_ALL_COND_METRICS = True
REF_ORDER = list(REF_CONFIGS.keys())

In [8]:
# =============================================================================
# 1. DATA LOADING & PREPROCESSING
# =============================================================================
print("=" * 80)
print("STEP 1: Data Loading & Preprocessing")
print("=" * 80)

preprocessor = GMpreprocess(file_path=DATASET_PATH, output_dir=PREPROCESS_OUTPUT_DIR)
data = preprocessor.run()
dataset_name = preprocessor.name

print(f"Dataset: {dataset_name}, shape: {data.shape}")
print(f"Time range: {data.index.min()} -> {data.index.max()}\n")

shared_pre = Urc1.preprocess_once(
    data,
    i_off=COMMON_CONFIG["i_off"],
    u_off=COMMON_CONFIG["u_off"],
    data_filter_i_min=COMMON_CONFIG["data_filter_i_min"],
    data_filter_U_min=COMMON_CONFIG["data_filter_U_min"],
    data_filter_U_max=COMMON_CONFIG["data_filter_U_max"],
    data_filter_T_min=COMMON_CONFIG["data_filter_T_min"],
    data_filter_T_max=COMMON_CONFIG["data_filter_T_max"],
)
print(f"Shared preprocessed rows: {len(shared_pre)}\n")

STEP 1: Data Loading & Preprocessing
=== 1. Loading & Preprocessing: G1M1_new ===
>> Data loaded successfully.
   [Detected] Temperature Col: 'Temp_Module_1' -> ID: '1'
>> Columns renamed to standard format.
>> Columns filtered. Retained: ['currentDensity', 'temperature', 'voltage']

=== 3. Saving Preprocessd data in .parquet format ===
>> ✅ Final Results saved successfully to:
   ..\\..\\explore_data\\output\G1M1_new_20260502_104503.parquet

=== GMpreprocess Pipeline Completed Successfully ===
Dataset: G1M1_new, shape: (2529217, 3)
Time range: 2021-02-12 00:00:00 -> 2025-12-09 09:59:00

Shared preprocessed rows: 1069062



In [9]:
# =============================================================================
# 2. TRAIN MODELS
# =============================================================================
print("=" * 80)
print("STEP 2: Model Training")
print("=" * 80)

models = {}

# ── 2a. Baseline (I²) ──
print("\n  -> Training Baseline (I²) model...")
urc_I2 = Urc1(
    data=data,
    name=dataset_name,
    preprocessed_data=shared_pre,
    **COMMON_CONFIG,
)
models["Baseline (I²)"] = urc_I2
print("  OK Baseline training completed")

# ── 2b. Cube-root ──
print("\n  -> Training Cube-root model...")
urc_cbrt = Urc1_c6_sigmoid(
    data=data,
    name=dataset_name,
    nonlinear_term="cbrt",
    preprocessed_data=shared_pre,
    **COMMON_CONFIG,
)
models["Cube-root"] = urc_cbrt
print("  OK Cube-root training completed")

# ── 2c. Sigmoid ──
print("\n  -> Training Sigmoid model...")
urc_sigmoid = Urc1_c6_sigmoid(
    data=data,
    name=dataset_name,
    nonlinear_term="sigmoid",
    preprocessed_data=shared_pre,
    **COMMON_CONFIG,
)
models["Sigmoid"] = urc_sigmoid
print("  OK Sigmoid training completed")

print(f"\nOK {len(models)} models trained successfully\n")

STEP 2: Model Training

  -> Training Baseline (I²) model...
  OK Baseline training completed

  -> Training Cube-root model...
Using shared preprocessed data (1069062 points, skipping preprocess).
Voltage model fitting (nonlinear_term='cbrt') ...
Fitting Stats: 474 intervals low data, 8 fit failed.


C:\Users\Z0057NPT\Documents\MA_code\degradation_toolbox\Urc\Urc1_c6_sigmoid.py:960: RuntimeWarning: invalid value encountered in sqrt
  
C:\Users\Z0057NPT\Documents\MA_code\degradation_toolbox\Urc\Urc1_c6_sigmoid.py:960: RuntimeWarning: invalid value encountered in sqrt
  


1042 out of 1766 fitting results are reliable.
  OK Cube-root training completed

  -> Training Sigmoid model...
Using shared preprocessed data (1069062 points, skipping preprocess).
Using sigmoid_steepness=5.000
Voltage model fitting (nonlinear_term='sigmoid') ...
Fitting Stats: 474 intervals low data, 3 fit failed.
359 out of 1766 fitting results are reliable.
  OK Sigmoid training completed

OK 3 models trained successfully



In [10]:
# =============================================================================
# 3. COMPARATOR SETUP & GT LOADING
# =============================================================================
print("\n" + "=" * 80)
print("STEP 3: Initialize Comparator & Load Ground Truth")
print("=" * 80)

comparator = UnifiedModelComparator(models)
print(f"OK UnifiedModelComparator initialized with {len(models)} models\n")

gt_loaded_count = 0
for ref_name, ref_cfg in REF_CONFIGS.items():
    iref = ref_cfg["Iref"]
    gt_file = ref_cfg["gt_file"]
    gt_path = Path(gt_file)
    if not gt_path.is_absolute():
        gt_path = Path.cwd() / gt_path
    gt_path = gt_path.resolve()

    if gt_path.exists():
        try:
            gt_data = pd.read_csv(gt_path, index_col=0, parse_dates=True)
            colmap = {str(c).strip().lower(): c for c in gt_data.columns}
            candidate_cols = ["gt_uref_regression", "voltage", "uref", "gt_uref"]
            selected_col = None
            for c in candidate_cols:
                if c in colmap:
                    selected_col = colmap[c]
                    break

            if selected_col is not None:
                gt_series = gt_data[selected_col]
            elif len(gt_data.columns) == 1:
                gt_series = gt_data.iloc[:, 0]
                selected_col = gt_data.columns[0]
            else:
                raise ValueError(
                    f"Cannot identify voltage column: {gt_data.columns.tolist()}"
                )

            gt_series = gt_series.dropna()
            comparator.set_ground_truth(gt_series, iref=iref)
            gt_loaded_count += 1
            print(
                f"  OK {ref_name} (Iref={iref}) GT loaded: {len(gt_series)} points [column: {selected_col}]"
            )
        except Exception as e:
            print(f"  FAILED {ref_name} GT: {e}")
    else:
        print(f"  MISSING GT file: {gt_path}")

has_gt = gt_loaded_count > 0 and SHOW_GT_METRICS
print(f"\nGT status: {gt_loaded_count}/{len(REF_CONFIGS)} references loaded")
print(f"GT metrics enabled: {has_gt}\n")


STEP 3: Initialize Comparator & Load Ground Truth
OK UnifiedModelComparator initialized with 3 models

  OK Low (Iref=0.3) GT loaded: 1762 points [column: gt_uref_regression]
  OK Medium (Iref=1.0) GT loaded: 1762 points [column: gt_uref_regression]
  OK High (Iref=1.48) GT loaded: 1384 points [column: gt_uref_regression]

GT status: 3/3 references loaded
GT metrics enabled: True



In [11]:
# =============================================================================
# 4. REFERENCE-SPECIFIC ANALYSIS (UNIFIED COMPARATOR)
# =============================================================================
print("=" * 80)
print("STEP 4: Reference-Specific Metrics & Comparison")
print("=" * 80)

rate_tables = []
for ref_name in REF_ORDER:
    ref_cfg = REF_CONFIGS[ref_name]
    iref = ref_cfg["Iref"]
    tref = ref_cfg["Tref"]
    ohref = ref_cfg["OHref"]

    print(f"\n### REFERENCE CONDITION: {ref_name} (Iref={iref}, Tref={tref}, OHref={ohref})")

    df_metrics = comparator.compare_all(
        i_target=iref,
        include_all_cond_metrics=SHOW_ALL_COND_METRICS,
        include_gt_metrics=has_gt,
    )

    if df_metrics.empty:
        print("No metrics returned for this reference.")
        continue

    display(Markdown(df_metrics.to_markdown(index=False)))

    rate_tables.append(
        df_metrics[["Model Name", "Target Current (A/cm2)", "Degradation Rate (uV/h)", "Slope Sigma (uV/h)"]].assign(
            Reference=ref_name
        )
    )

    try:
        comparator.plot_interactive_trends(
            target_i=iref,
            show_gt=has_gt,
            save=SAVE_PLOTS,
            output_dir=PLOTS_OUTPUT_DIR,
        )
        print("OK Trend plot finished")
    except Exception as e:
        print(f"Trend plot failed: {e}")

    comparator.print_comparison_report(
        i_target=iref,
        include_gt_metrics=has_gt,
        include_all_cond_metrics=SHOW_ALL_COND_METRICS,
    )

STEP 4: Reference-Specific Metrics & Comparison

### REFERENCE CONDITION: Low (Iref=0.3, Tref=58, OHref=11)


| Model Name    |   Target Current (A/cm2) |   Fitting Time (s) |   Data Points (n) |   Degradation Rate (uV/h) |   RMSE (mV) |   Slope Sigma (uV/h) |   Mono (Rank) [0-1] |   Outlier Count |   Outlier (%) |   Max Residual (mV) |   Mean SE (mV) |   Cond Median log10 |   Cond P95 log10 |   Cond Count (Used Scope) | Cond Scope   |   GT RMSE (mV) |   GT MAE (mV) |   GT Valid Intervals |
|:--------------|-------------------------:|-------------------:|------------------:|--------------------------:|------------:|---------------------:|--------------------:|----------------:|--------------:|--------------------:|---------------:|--------------------:|-----------------:|--------------------------:|:-------------|---------------:|--------------:|---------------------:|
| Baseline (I²) |                      0.3 |             15.727 |              1098 |                   1.25211 |      18.885 |                0     |               0.925 |              10 |          0.91 |             390.861 |          4.105 |               4.745 |           13.287 |                      1286 | all_fitted   |         19.114 |         5.783 |                 1098 |
| Cube-root     |                      0.3 |            315.323 |              1042 |                   1.18266 |      25.539 |                0.02  |               0.916 |               9 |          0.86 |             604.616 |          0.505 |               4.809 |           16.555 |                      1278 | all_fitted   |         25.774 |         6.359 |                 1042 |
| Sigmoid       |                      0.3 |            232.123 |               359 |                   1.05147 |      23.491 |                0.047 |               0.862 |               4 |          1.11 |             376.554 |          0.502 |              12.182 |           17.83  |                      1283 | all_fitted   |         23.627 |         7.047 |                  359 |

OK Trend plot finished
  MODEL COMPARISON REPORT @ 0.3 A/cm²

📊 Baseline (I²)
------------------------------------------------------------
  Data Points:      1098
  Deg Rate:         1.252113 μV/h
  RMSE:             18.885 mV
  Slope Sigma:      0.000 μV/h
  Mono (Rank):      0.925
  Outliers:         10 (0.9%)
  Max Residual:     390.861 mV
  Mean SE:          4.105 mV
  Cond (Median):    10^4.7
  Cond Scope:       all_fitted (n=1286)
  GT RMSE:          19.114 mV (n=1098)
  GT MAE:           5.783 mV

📊 Cube-root
------------------------------------------------------------
  Data Points:      1042
  Deg Rate:         1.182656 μV/h
  RMSE:             25.539 mV
  Slope Sigma:      0.020 μV/h
  Mono (Rank):      0.916
  Outliers:         9 (0.9%)
  Max Residual:     604.616 mV
  Mean SE:          0.505 mV
  Cond (Median):    10^4.8
  Cond Scope:       all_fitted (n=1278)
  GT RMSE:          25.774 mV (n=1042)
  GT MAE:           6.359 mV

📊 Sigmoid
-----------------------------------

| Model Name    |   Target Current (A/cm2) |   Fitting Time (s) |   Data Points (n) |   Degradation Rate (uV/h) |   RMSE (mV) |   Slope Sigma (uV/h) |   Mono (Rank) [0-1] |   Outlier Count |   Outlier (%) |   Max Residual (mV) |   Mean SE (mV) |   Cond Median log10 |   Cond P95 log10 |   Cond Count (Used Scope) | Cond Scope   |   GT RMSE (mV) |   GT MAE (mV) |   GT Valid Intervals |
|:--------------|-------------------------:|-------------------:|------------------:|--------------------------:|------------:|---------------------:|--------------------:|----------------:|--------------:|--------------------:|---------------:|--------------------:|-----------------:|--------------------------:|:-------------|---------------:|--------------:|---------------------:|
| Baseline (I²) |                        1 |             15.727 |              1098 |                   5.20858 |      18.873 |                0     |               0.959 |              41 |          3.73 |             141.761 |          4.137 |               4.745 |           13.287 |                      1286 | all_fitted   |         20.604 |        15.377 |                 1098 |
| Cube-root     |                        1 |            315.323 |              1042 |                   5.7259  |      21.694 |                0.037 |               0.955 |              30 |          2.88 |             252.079 |          0.805 |               4.809 |           16.555 |                      1278 | all_fitted   |         22.395 |        16.314 |                 1042 |
| Sigmoid       |                        1 |            232.123 |               359 |                   6.54526 |      25.202 |                0.085 |               0.964 |              18 |          5.01 |             235.085 |          0.807 |              12.182 |           17.83  |                      1283 | all_fitted   |         22.204 |        12.751 |                  359 |

OK Trend plot finished
  MODEL COMPARISON REPORT @ 1.0 A/cm²

📊 Baseline (I²)
------------------------------------------------------------
  Data Points:      1098
  Deg Rate:         5.208581 μV/h
  RMSE:             18.873 mV
  Slope Sigma:      0.000 μV/h
  Mono (Rank):      0.959
  Outliers:         41 (3.7%)
  Max Residual:     141.761 mV
  Mean SE:          4.137 mV
  Cond (Median):    10^4.7
  Cond Scope:       all_fitted (n=1286)
  GT RMSE:          20.604 mV (n=1098)
  GT MAE:           15.377 mV

📊 Cube-root
------------------------------------------------------------
  Data Points:      1042
  Deg Rate:         5.725895 μV/h
  RMSE:             21.694 mV
  Slope Sigma:      0.037 μV/h
  Mono (Rank):      0.955
  Outliers:         30 (2.9%)
  Max Residual:     252.079 mV
  Mean SE:          0.805 mV
  Cond (Median):    10^4.8
  Cond Scope:       all_fitted (n=1278)
  GT RMSE:          22.395 mV (n=1042)
  GT MAE:           16.314 mV

📊 Sigmoid
--------------------------------

| Model Name    |   Target Current (A/cm2) |   Fitting Time (s) |   Data Points (n) |   Degradation Rate (uV/h) |   RMSE (mV) |   Slope Sigma (uV/h) |   Mono (Rank) [0-1] |   Outlier Count |   Outlier (%) |   Max Residual (mV) |   Mean SE (mV) |   Cond Median log10 |   Cond P95 log10 |   Cond Count (Used Scope) | Cond Scope   |   GT RMSE (mV) |   GT MAE (mV) |   GT Valid Intervals |
|:--------------|-------------------------:|-------------------:|------------------:|--------------------------:|------------:|---------------------:|--------------------:|----------------:|--------------:|--------------------:|---------------:|--------------------:|-----------------:|--------------------------:|:-------------|---------------:|--------------:|---------------------:|
| Baseline (I²) |                     1.48 |             15.727 |              1098 |                  10.9523  |      55.143 |                0     |               0.914 |              65 |          5.92 |             326.556 |          5.353 |               4.745 |           13.287 |                      1286 | all_fitted   |         64.473 |        48.601 |                  868 |
| Cube-root     |                     1.48 |            315.323 |              1042 |                   9.91746 |      42.382 |                0.077 |               0.929 |              47 |          4.51 |             235.445 |          1.767 |               4.809 |           16.555 |                      1278 | all_fitted   |         50.43  |        35.152 |                  817 |
| Sigmoid       |                     1.48 |            232.123 |               359 |                  10.9719  |      59.796 |                0.285 |               0.926 |              23 |          6.41 |             384.351 |          4.8   |              12.182 |           17.83  |                      1283 | all_fitted   |         52.716 |        34.703 |                  277 |

OK Trend plot finished
  MODEL COMPARISON REPORT @ 1.48 A/cm²

📊 Baseline (I²)
------------------------------------------------------------
  Data Points:      1098
  Deg Rate:         10.952308 μV/h
  RMSE:             55.143 mV
  Slope Sigma:      0.000 μV/h
  Mono (Rank):      0.914
  Outliers:         65 (5.9%)
  Max Residual:     326.556 mV
  Mean SE:          5.353 mV
  Cond (Median):    10^4.7
  Cond Scope:       all_fitted (n=1286)
  GT RMSE:          64.473 mV (n=868)
  GT MAE:           48.601 mV

📊 Cube-root
------------------------------------------------------------
  Data Points:      1042
  Deg Rate:         9.917462 μV/h
  RMSE:             42.382 mV
  Slope Sigma:      0.077 μV/h
  Mono (Rank):      0.929
  Outliers:         47 (4.5%)
  Max Residual:     235.445 mV
  Mean SE:          1.767 mV
  Cond (Median):    10^4.8
  Cond Scope:       all_fitted (n=1278)
  GT RMSE:          50.430 mV (n=817)
  GT MAE:           35.152 mV

📊 Sigmoid
--------------------------------

In [12]:
# =============================================================================
# 5. CROSS-MODEL DIAGNOSTICS + SUMMARY
# =============================================================================
print("\n" + "=" * 80)
print("STEP 5: Cross-Model Diagnostics")
print("=" * 80)

try:
    print("\n-> Plotting fit quality (RMSE & R2 distributions)...")
    comparator.plot_fit_quality(save=SAVE_PLOTS)
    print("OK Fit quality completed")
except Exception as e:
    print(f"Fit quality plot failed: {e}")

try:
    print("\n-> Plotting coefficient diagnostics (include c6)...")
    comparator.plot_coefficient_diagnostic(save=SAVE_PLOTS, include_c6=True)
    print("OK Coefficient diagnostic completed")
except Exception as e:
    print(f"Coefficient diagnostic failed: {e}")

try:
    print("\n-> Plotting coverage Gantt...")
    comparator.plot_coverage_gantt(save=SAVE_PLOTS)
    print("OK Coverage Gantt completed")
except Exception as e:
    print(f"Coverage Gantt failed: {e}")

print("\n" + "=" * 80)
print("ANALYSIS COMPLETE")
print("=" * 80)
print(f"\nModels trained: {len(models)}")
print(f"Reference conditions analyzed: {len(REF_CONFIGS)}")
print(f"Ground truth loaded: {has_gt}")
print(f"Output plots dir: {PLOTS_OUTPUT_DIR}" if SAVE_PLOTS else "Plots displayed only")
print("\nKey settings:")
print(f"  - include_all_cond_metrics: {SHOW_ALL_COND_METRICS}")
print(f"  - include_gt_metrics: {SHOW_GT_METRICS}")

if rate_tables:
    print("\nDegradation-rate summary across references:")
    display(pd.concat(rate_tables, ignore_index=True))


STEP 5: Cross-Model Diagnostics

-> Plotting fit quality (RMSE & R2 distributions)...
OK Fit quality completed

-> Plotting coefficient diagnostics (include c6)...
OK Coefficient diagnostic completed

-> Plotting coverage Gantt...
OK Coverage Gantt completed

ANALYSIS COMPLETE

Models trained: 3
Reference conditions analyzed: 3
Ground truth loaded: True
Output plots dir: ..\\plots\\c6\\G1M1_comparison

Key settings:
  - include_all_cond_metrics: True
  - include_gt_metrics: True

Degradation-rate summary across references:


,Model Name,Target Current (A/cm2),Degradation Rate (uV/h),Slope Sigma (uV/h),Reference
0,Baseline (I²),0.30,1.252113,0.000,Low
1,Cube-root,0.30,1.182656,0.020,Low
2,Sigmoid,0.30,1.051471,0.047,Low
3,Baseline (I²),1.00,5.208581,0.000,Medium
4,Cube-root,1.00,5.725895,0.037,Medium
5,Sigmoid,1.00,6.545263,0.085,Medium
6,Baseline (I²),1.48,10.952308,0.000,High
7,Cube-root,1.48,9.917462,0.077,High
8,Sigmoid,1.48,10.971924,0.285,High
